# Exploratory Data Analysis (EDA)## AI-Powered Sustainable Transportation Advisor**Objective:** Explore the Delhi bus GPS dataset to understand data volume, quality, temporal patterns, and route geography — informing our ML pipeline design decisions.

In [ ]:
import sqlite3from pathlib import Pathimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as sns# Set plot aestheticssns.set_theme(style="whitegrid", palette="muted")plt.rcParams["figure.figsize"] = (10, 5)plt.rcParams["font.size"] = 10# Resolve paths relative to repository rootPROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()DB_PATH = (PROJECT_ROOT / "data" / "raw" / "2019-07-01.db").resolve()print(f"Project Root : {PROJECT_ROOT}")print(f"Database Path: {DB_PATH}")assert DB_PATH.exists(), f"Database file not found at {DB_PATH}"def get_db():    return sqlite3.connect(f"{DB_PATH.as_uri()}?mode=ro", uri=True)

---## Dataset Overview & ScaleWe examine the total number of records, distinct vehicles, routes, and trips to understand the scope of the July 1, 2019 dataset.

In [ ]:
with get_db() as conn:    overview_query = """    SELECT         COUNT(*) AS total_records,        COUNT(DISTINCT vehicle_id) AS distinct_vehicles,        COUNT(DISTINCT route_id) AS distinct_routes,        COUNT(DISTINCT trip_id) AS distinct_trips,        MIN(timestamp) AS min_utc_epoch,        MAX(timestamp) AS max_utc_epoch,        MIN(time) AS min_server_time,        MAX(time) AS max_server_time    FROM vehicle_feed;    """    overview_df = pd.read_sql_query(overview_query, conn)print("=== Dataset Overview (July 1, 2019) ===")for col in overview_df.columns:    print(f"- {col:20s}: {overview_df[col].iloc[0]}")

> **Outcome:** The dataset covers **~2 million GPS pings** from **816 buses** across **193 routes** > in a single day. This confirms sufficient data volume for training travel-time prediction models.

---## Data Quality & Deduplication AuditWe check for duplicate records — same bus, same timestamp — to determine how much data cleaning is needed before feature extraction.

In [ ]:
with get_db() as conn:    c = conn.cursor()        # 1. Total duplicate (vehicle_id, timestamp) pairs    dup_key_pings = c.execute("""        SELECT COUNT(*) - COUNT(DISTINCT vehicle_id || '_' || timestamp)        FROM vehicle_feed;    """).fetchone()[0]        # 2. Conflicting coordinates among duplicate keys    conflicting_coords = c.execute("""        SELECT COUNT(*) FROM (            SELECT vehicle_id, timestamp            FROM vehicle_feed            GROUP BY vehicle_id, timestamp            HAVING COUNT(DISTINCT lat) > 1 OR COUNT(DISTINCT lng) > 1        );    """).fetchone()[0]        # 3. Conflicting speed or route among duplicate keys    conflicting_speed = c.execute("""        SELECT COUNT(*) FROM (            SELECT vehicle_id, timestamp            FROM vehicle_feed            GROUP BY vehicle_id, timestamp            HAVING COUNT(DISTINCT speed) > 1        );    """).fetchone()[0]        conflicting_route = c.execute("""        SELECT COUNT(*) FROM (            SELECT vehicle_id, timestamp            FROM vehicle_feed            GROUP BY vehicle_id, timestamp            HAVING COUNT(DISTINCT route_id) > 1        );    """).fetchone()[0]print("=== Deduplication & Collision Audit ===")print(f"- Duplicate (vehicle_id, timestamp) pings: {dup_key_pings:,} ({(dup_key_pings / 1977922)*100:.2f}% of records)")print(f"- Duplicate keys with conflicting coordinates : {conflicting_coords}")print(f"- Duplicate keys with conflicting speeds      : {conflicting_speed}")print(f"- Duplicate keys with conflicting route IDs   : {conflicting_route}")print("\n[Finding] Zero conflicts exist across all duplicate keys. Duplicate rows are redundant server-side retransmissions.")print(f"- Effective unique kinematic observations: {1977922 - dup_key_pings:,}")

> **Outcome:** Found **73,034 duplicate** (vehicle_id, timestamp) pairs with **zero coordinate conflicts** — > meaning deduplication is safe and removes only truly redundant rows. > After deduplication: **1,904,888 unique observations** remain.

---## Temporal Sampling & Transmission DelayWe analyze how frequently GPS pings arrive (sampling interval) and the delay between the bus's onboard clock and the central server's reception time.

In [ ]:
with get_db() as conn:    # Sample active trajectory for Route 90 (the highest-volume route)    traj_df = pd.read_sql_query("""        SELECT vehicle_id, CAST(timestamp AS INTEGER) as ts_utc, time, date        FROM vehicle_feed        WHERE route_id = 90.0        ORDER BY vehicle_id, ts_utc ASC;    """, conn)# Deduplicate exact (vehicle_id, ts_utc) pairstraj_clean = traj_df.drop_duplicates(subset=["vehicle_id", "ts_utc"]).copy()traj_clean["dt"] = traj_clean.groupby("vehicle_id")["ts_utc"].diff()# 1. Sampling Interval Metricsdt_series = traj_clean["dt"].dropna()print("=== Sampling Interval (dt) Percentiles (Route 90) ===")print(dt_series.describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]))print(f"- dt <= 15s : {(dt_series <= 15).mean() * 100:.2f}%")print(f"- dt <= 30s : {(dt_series <= 30).mean() * 100:.2f}%")print(f"- dt <= 60s : {(dt_series <= 60).mean() * 100:.2f}%")# 2. Transmission Latency Metrics# IST midnight was 1561919400ist_midnight = 1561919400traj_clean["onboard_ist_sec"] = traj_clean["ts_utc"] - ist_midnightdef parse_sec(t_str):    p = t_str.split(":")    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])traj_clean["server_sec"] = traj_clean["time"].apply(parse_sec)traj_clean["latency_sec"] = traj_clean["server_sec"] - traj_clean["onboard_ist_sec"]print("\n=== Transmission Latency (Server Ingestion vs. Onboard Clock) ===")print(traj_clean["latency_sec"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

In [ ]:
# Visualizing Sampling Interval & Transmission Latencyfig, axes = plt.subplots(1, 2, figsize=(14, 5))# Plot 1: Sampling interval distribution (clipped to <= 60s for visualization)sns.histplot(dt_series[dt_series <= 60], bins=30, ax=axes[0], color="royalblue", kde=True)axes[0].set_title("GPS Sampling Interval Distribution (dt <= 60s)")axes[0].set_xlabel("Delta t (seconds)")axes[0].set_ylabel("Observation Count")axes[0].axvline(10, color="red", linestyle="--", label="Median: 10s")axes[0].legend()# Plot 2: Transmission latency distribution (clipped to <= 120s)lat_clipped = traj_clean["latency_sec"][traj_clean["latency_sec"].between(0, 120)]sns.histplot(lat_clipped, bins=40, ax=axes[1], color="darkorange", kde=True)axes[1].set_title("Cellular Transmission Latency Distribution (<= 120s)")axes[1].set_xlabel("Transmission Delay (Server Time - Onboard Time in seconds)")axes[1].set_ylabel("Observation Count")axes[1].axvline(traj_clean["latency_sec"].median(), color="red", linestyle="--", label=f"Median: {traj_clean['latency_sec'].median():.0f}s")axes[1].legend()plt.tight_layout()plt.show()

> **Outcome:** GPS pings arrive with a **median interval of ~10 seconds**. > Server transmission delay has a **median of ~13 seconds** with occasional spikes up to 3 minutes. > This confirms we should use the onboard `timestamp` (not server `time`) for trajectory ordering.

---## Route Corridor Selection & Trajectory MappingWe identify the busiest routes and map their GPS trajectories to select suitable road corridors for our travel-time prediction model.

In [ ]:
with get_db() as conn:    top_routes_df = pd.read_sql_query("""        SELECT             route_id,            COUNT(*) AS total_pings,            COUNT(DISTINCT vehicle_id) AS n_buses,            COUNT(DISTINCT trip_id) AS n_trips,            MIN(CAST(lat AS REAL)) AS min_lat,            MAX(CAST(lat AS REAL)) AS max_lat,            MIN(CAST(lng AS REAL)) AS min_lng,            MAX(CAST(lng AS REAL)) AS max_lng        FROM vehicle_feed        GROUP BY route_id        ORDER BY total_pings DESC        LIMIT 10;    """, conn)top_routes_df["route_id"] = top_routes_df["route_id"].apply(lambda x: str(int(x)))display(top_routes_df)

In [ ]:
# Spatial Trajectory Mapping for Candidate Route 90 and Route 399with get_db() as conn:    sample_geo = pd.read_sql_query("""        SELECT route_id, vehicle_id, CAST(lat AS REAL) as lat, CAST(lng AS REAL) as lng        FROM vehicle_feed        WHERE route_id IN (90.0, 399.0)        LIMIT 15000;    """, conn)sample_geo["route_label"] = sample_geo["route_id"].apply(lambda x: f"Route {int(x)}")plt.figure(figsize=(10, 8))sns.scatterplot(    data=sample_geo,    x="lng",    y="lat",    hue="route_label",    alpha=0.3,    s=12,    palette=["#1f77b4", "#ff7f0e"])plt.title("Spatial GPS Observation Trajectories: Candidate Routes 90 & 399")plt.xlabel("Longitude (°E)")plt.ylabel("Latitude (°N)")plt.legend(title="Corridor")plt.show()

> **Outcome:** **Route 90** (Outer Ring Road) is the highest-volume route with the most GPS pings. > Route 399 (MB Road) provides a second corridor for cross-validation. > The trajectory maps confirm clean, well-defined spatial corridors suitable for gate-based segment definition.

---## Key EDA Conclusions| Finding | Implication for ML Pipeline ||---------|---------------------------|| ~2M GPS pings, 816 buses, 193 routes | Sufficient training data volume || 73K duplicate rows (safe to remove) | Apply deduplication as preprocessing step || 10-second sampling interval | Dense enough for trajectory reconstruction || Server delay up to 3 min | Use onboard timestamp, not server time || Route 90 highest-volume corridor | Selected as primary prediction target |These findings directly inform the **data cleaning**, **label construction**, and **feature engineering** phases.